In [ ]:

!pip uninstall -y jax jaxlib ml-dtypes ml_dtypes tensorstore dopamine-rl >/dev/null 2>&1 || echo "Some removals may not be needed"
!pip install --force-reinstall -q "tensorflow==2.19.0"
!pip install -q pyarrow fastparquet imbalanced-learn scikit-learn pandas numpy joblib

import tensorflow as tf
print("TensorFlow:", tf.__version__)

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from google.colab import files
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, f1_score
from imblearn.over_sampling import BorderlineSMOTE
from tensorflow.keras import layers, models

print("Upload 'UNSW_NB15_training-set.parquet' (required). Optionally upload 'UNSW_NB15_testing-set.parquet'.")
uploaded = files.upload()


train_file = None
test_file = None
for fn in uploaded.keys():
    lname = fn.lower()
    if "training" in lname:
        train_file = fn
    if "testing" in lname:
        test_file = fn

if not train_file:
    raise FileNotFoundError("Training parquet file not found in uploads. Please upload UNSW_NB15_training-set.parquet.")

has_test = test_file is not None
print("Training:", train_file, "| Testing provided:", has_test)

df_train = pd.read_parquet(train_file)
df_test = pd.read_parquet(test_file) if has_test else None
print("Loaded shapes -> train:", df_train.shape, " test:", None if df_test is None else df_test.shape)

# Basic cleaning & label unify
def clean(df):
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
    if "label" not in df.columns and "attack_cat" in df.columns:
        df["label"] = (df["attack_cat"] != "Normal").astype(int)
    if "label" not in df.columns:
        raise ValueError("No 'label' or 'attack_cat' column found in dataframe.")
    df["label"] = df["label"].astype(int)
    return df

df_train = clean(df_train)
if has_test:
    df_test = clean(df_test)

print("Label distribution (train):", np.bincount(df_train["label"]))

# Convert categorical Pandas Categorical -> str to avoid fillna/category issues
categorical_candidates = [c for c in ["state","service","proto"] if c in df_train.columns]
for c in categorical_candidates:
    df_train[c] = df_train[c].astype(str)
    if has_test:
        df_test[c] = df_test[c].astype(str)

# OneHotEncoder with compatibility for sklearn versions
if len(categorical_candidates) > 0:
    # handle parameter name differences
    try:

        enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    except TypeError:

        enc = OneHotEncoder(sparse=False, handle_unknown="ignore")
    enc.fit(df_train[categorical_candidates])
    train_ohe = enc.transform(df_train[categorical_candidates])
    df_train = pd.concat([
        df_train.drop(columns=categorical_candidates),
        pd.DataFrame(train_ohe, columns=enc.get_feature_names_out(), index=df_train.index)
    ], axis=1)
    if has_test:
        test_ohe = enc.transform(df_test[categorical_candidates])
        df_test = pd.concat([
            df_test.drop(columns=categorical_candidates),
            pd.DataFrame(test_ohe, columns=enc.get_feature_names_out(), index=df_test.index)
        ], axis=1)
    print("One-hot encoded. New train shape:", df_train.shape)

# Standard scaling and feature list
exclude_cols = ["label","attack_cat"]
feature_cols = [c for c in df_train.columns if c not in exclude_cols]
print("Total features before selection:", len(feature_cols))

scaler = StandardScaler()
df_train[feature_cols] = scaler.fit_transform(df_train[feature_cols])
if has_test:
    df_test[feature_cols] = scaler.transform(df_test[feature_cols])

# BRFE-like feature selection: RandomForest importances -> top K (K=23)
K = 23
rf_tmp = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_tmp.fit(df_train[feature_cols], df_train["label"])
feat_imp = pd.Series(rf_tmp.feature_importances_, index=feature_cols).sort_values(ascending=False)
topK = feat_imp.head(K).index.tolist()
print("Selected top-K features (K=%d):" % K, topK)

# Sliding-window creation (window size = 5)
WINDOW = 5
def sliding_windows(df, features, window=WINDOW):
    X, y = [], []
    arr = df[features].values
    labs = df["label"].values
    for i in range(len(df) - window + 1):
        X.append(arr[i:i+window])
        y.append(int(labs[i:i+window].max() > 0))
    return np.array(X), np.array(y)

if has_test:
    X_train_w, y_train_w = sliding_windows(df_train.reset_index(drop=True), topK, window=WINDOW)
    X_test_w,  y_test_w  = sliding_windows(df_test.reset_index(drop=True),  topK, window=WINDOW)
else:
    X_all, y_all = sliding_windows(df_train.reset_index(drop=True), topK, window=WINDOW)
    X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=42)

print("Windowed shapes:", X_train_w.shape, X_test_w.shape)
print("Window label distribution (train):", np.bincount(y_train_w))

# Borderline-SMOTE on flattened windows
X_train_flat = X_train_w.reshape(len(X_train_w), -1)
print("Applying Borderline-SMOTE (may take a short while)...")
sm = BorderlineSMOTE()
X_res, y_res = sm.fit_resample(X_train_flat, y_train_w)
X_res_w = X_res.reshape(-1, WINDOW, len(topK))
print("After resample:", X_res_w.shape, "Positive ratio:", y_res.mean())

# RandomForest baseline (flattened windows)
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_res, y_res)
rf_pred = rf.predict(X_test_w.reshape(len(X_test_w), -1))

rf_result = {
    "Model": "RandomForest",
    "Accuracy": float(accuracy_score(y_test_w, rf_pred)),
    "Precision": float(precision_score(y_test_w, rf_pred, zero_division=0)),
    "F1": float(f1_score(y_test_w, rf_pred, zero_division=0))
}
print("RandomForest baseline:", rf_result)

# Model builders (CNN, CNN-LSTM, CNN-BiLSTM)
def build_cnn(input_shape):
    inp = layers.Input(shape=input_shape)
    x = layers.Conv1D(64, 3, activation="relu", padding="same")(inp)
    x = layers.Conv1D(128, 3, activation="relu", padding="same")(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(2, activation="softmax")(x)
    return models.Model(inp, out)

def build_cnn_lstm(input_shape):
    inp = layers.Input(shape=input_shape)
    x = layers.Conv1D(64, 3, activation="relu")(inp)
    x = layers.MaxPool1D(2)(x)
    x = layers.LSTM(64)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(2, activation="softmax")(x)
    return models.Model(inp, out)

def build_cnn_bilstm(input_shape):
    inp = layers.Input(shape=input_shape)
    x = layers.Conv1D(64, 3, activation="relu")(inp)
    x = layers.MaxPool1D(2)(x)
    x = layers.Bidirectional(layers.LSTM(64))(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(2, activation="softmax")(x)
    return models.Model(inp, out)

# Shape-safe TBT (project input to 128 dims first)
def build_tbt(input_shape):
    inp = layers.Input(shape=input_shape)
    proj = layers.Dense(128)(inp)
    def tcn_block_local(x, filters=128):
        for d in [1,2,4]:
            res = x
            y = layers.Conv1D(filters, 3, padding="causal", dilation_rate=d, activation="relu")(x)
            y = layers.Conv1D(filters, 3, padding="causal", dilation_rate=d, activation="relu")(y)
            x = layers.Add()([res, y])
        return x

    t = tcn_block_local(proj)
    t = layers.GlobalAveragePooling1D()(t)
    b = layers.Bidirectional(layers.LSTM(128))(proj)
    att = layers.MultiHeadAttention(num_heads=4, key_dim=32)(proj, proj)
    att = layers.LayerNormalization()(proj + att)
    ff = layers.Dense(256, activation="relu")(att)
    ff = layers.Dense(128)(ff)
    att = layers.LayerNormalization()(att + ff)
    att = layers.GlobalAveragePooling1D()(att)


    concat = layers.Concatenate()([t, b, att])
    x = layers.Dense(256, activation="relu")(concat)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(2, activation="softmax")(x)

    return models.Model(inp, out)


EPOCHS = 5
BATCH = 64

def train_and_eval(model, X_tr, y_tr, X_val, y_val, name, epochs=EPOCHS):
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=epochs, batch_size=BATCH, verbose=1)
    preds = model.predict(X_val).argmax(axis=1)
    return {
        "Model": name,
        "Accuracy": float(accuracy_score(y_val, preds)),
        "Precision": float(precision_score(y_val, preds, zero_division=0)),
        "F1": float(f1_score(y_val, preds, zero_division=0))
    }

# Build and train models
input_shape = (WINDOW, len(topK))
print("Model input shape:", input_shape)

print("Training CNN...")
cnn_model = build_cnn(input_shape)
res_cnn = train_and_eval(cnn_model, X_res_w, y_res, X_test_w, y_test_w, "CNN")

print("Training CNN-LSTM...")
cnn_lstm_model = build_cnn_lstm(input_shape)
res_cnn_lstm = train_and_eval(cnn_lstm_model, X_res_w, y_res, X_test_w, y_test_w, "CNN-LSTM")

print("Training CNN-BiLSTM...")
cnn_bi_model = build_cnn_bilstm(input_shape)
res_cnn_bi = train_and_eval(cnn_bi_model, X_res_w, y_res, X_test_w, y_test_w, "CNN-BiLSTM")

print("Training TBT (shape-safe)...")
tbt_model = build_tbt(input_shape)
res_tbt = train_and_eval(tbt_model, X_res_w, y_res, X_test_w, y_test_w, "TBT")

#Final results table
import pandas as pd
results = pd.DataFrame([rf_result, res_cnn, res_cnn_lstm, res_cnn_bi, res_tbt])
print("\nFinal results:")
print(results)
results


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
optax 0.2.6 requires jax>=0.5.3, which is not installed.
optax 0.2.6 requires jaxlib>=0.5.3, which is not installed.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
orbax-checkpoint 0.11.28 requires jax>=0.6.0, which is not installed.
orbax-checkpoint 0.11.28 requires tensorstore>=0.1.71, which is not installed.
flax 0.10.7 requires jax>=0.6.0, which is not installed.
flax 0.10.7 requires tensorstore, which is not installed.
chex 0.1.90 requires jax>=0.4.27, which is not installed.
chex 0.1.90 requires jaxlib>=0.4.27, which is not installed.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.1.3 which is incompatible.
bigframes 2.29.1 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is i

Saving UNSW_NB15_training-set.parquet to UNSW_NB15_training-set (1).parquet
Training: UNSW_NB15_training-set (1).parquet | Testing provided: False
Loaded shapes -> train: (175341, 36)  test: None
Label distribution (train): [ 56000 119341]
One-hot encoded. New train shape: (175341, 188)
Total features before selection: 186
Selected top-K features (K=23): ['rate', 'tcprtt', 'dload', 'ackdat', 'sload', 'synack', 'dinpkt', 'dur', 'dmean', 'sbytes', 'sinpkt', 'dbytes', 'smean', 'dpkts', 'state_INT', 'sjit', 'djit', 'spkts', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'dloss', 'state_CON', 'sloss']
Windowed shapes: (140269, 5, 23) (35068, 5, 23)
Window label distribution (train): [ 38631 101638]
Applying Borderline-SMOTE (may take a short while)...
After resample: (203276, 5, 23) Positive ratio: 0.5
RandomForest baseline: {'Model': 'RandomForest', 'Accuracy': 0.988279913311281, 'Precision': 0.9925134954095906, 'F1': 0.9919076965484652}
Model input shape: (5, 23)
Training CNN...
Epoch 1/5
3177/3

,Model,Accuracy,Precision,F1
0,RandomForest,0.988280,0.992513,0.991908
1,CNN,0.981179,0.997228,0.986878
2,CNN-LSTM,0.968689,0.993224,0.978065
3,CNN-BiLSTM,0.964098,0.995243,0.974715
4,TBT,0.986740,0.997051,0.990793
